# Model Evaluation on Teacher Datasets

This notebook loads pre-trained models and evaluates them on the teacher datasets:
1. Load pre-trained models from '../../trained_models/tensorflow/'
2. Load teacher datasets from '../../datasets/'
3. Preprocess the text data for each model
4. Make predictions with each model
5. Compare predictions with actual labels
6. Calculate and visualize accuracy metrics


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
import re
import pickle
import glob
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

np.random.seed(2025)
tf.random.set_seed(2025)


## Define Preprocessing Functions

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    
    text = re.sub(r"https?://\S+|www\.\S+|\S+@\S+\.\S+", "", text)
    
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

def extract_features(texts):
    print("Extracting features...")
    features = []
    stop_words = set(stopwords.words("english"))
    
    for text in texts:
        if not isinstance(text, str) or not text.strip():
            features.append([0] * 8)
            continue
            
        words = word_tokenize(text)
        
        if not words:
            features.append([0] * 8)
            continue
            
        num_words = len(words)
        num_chars = len(text)
        avg_word_length = np.mean([len(w) for w in words]) if words else 0
        num_stopwords = sum(1 for word in words if word.lower() in stop_words)
        stopword_ratio = num_stopwords / num_words if num_words else 0
        
        num_punctuation = sum(1 for char in text if char in string.punctuation)
        punctuation_ratio = num_punctuation / num_chars if num_chars else 0
        
        lexical_diversity = len(set(words)) / num_words if num_words else 0
        
        feature_vector = [
            num_words,
            num_chars,
            avg_word_length,
            num_stopwords,
            stopword_ratio,
            num_punctuation,
            punctuation_ratio,
            lexical_diversity
        ]
        
        features.append(feature_vector)
        
    return np.array(features)


## Load Datasets

In [ ]:

teacher_inputs_path = "../datasets/clean/teacher_inputs.csv"
teacher_inputs = pd.read_csv(teacher_inputs_path, sep="\t")
print(f"Teacher inputs shape: {teacher_inputs.shape}")
print(teacher_inputs.head())

teacher_outputs_path = "../datasets/clean/teacher_outputs.csv"
teacher_outputs = pd.read_csv(teacher_outputs_path, sep="\t")
print(f"\nTeacher outputs shape: {teacher_outputs.shape}")
print(teacher_outputs.head())

teacher_data = pd.merge(teacher_inputs, teacher_outputs, on="ID")
print(f"\nMerged data shape: {teacher_data.shape}")
print(teacher_data.head())

teacher_data['numeric_label'] = teacher_data['Label'].apply(lambda x: 0 if x == 'Human' else 1)

print(f"\nLabel distribution: {teacher_data['Label'].value_counts().to_dict()}")


## Load Pre-trained Models

In [ ]:

model_dir = "../trained_models/tensorflow/"

if not os.path.exists(model_dir):
    print(f"Model directory {model_dir} not found. Creating it.")
    os.makedirs(model_dir, exist_ok=True)
    print("No models found. Please train models first.")
else:
    print(f"Looking for models in {model_dir}")

model_files = glob.glob(os.path.join(model_dir, "*.h5"))
print(f"Found {len(model_files)} model files: {[os.path.basename(f) for f in model_files]}")

preprocessing_files = glob.glob(os.path.join(model_dir, "*.pkl"))
print(f"Found {len(preprocessing_files)} preprocessing files: {[os.path.basename(f) for f in preprocessing_files]}")

models = {}
preprocessors = {}

for model_file in model_files:
    model_name = os.path.basename(model_file).replace("_model.h5", "")
    
    try:
        print(f"Loading model: {model_name}")
        models[model_name] = keras.models.load_model(model_file)
        
        preprocessor_file = None
        for prep_file in preprocessing_files:
            if model_name in prep_file:
                preprocessor_file = prep_file
                break
        
        if preprocessor_file:
            print(f"Loading preprocessor: {os.path.basename(preprocessor_file)}")
            try:
                with open(preprocessor_file, 'rb') as f:
                    preprocessor = pickle.load(f)
                
                if "dnn" in model_name:
                    if isinstance(preprocessor, dict):
                        if 'extract_features' not in preprocessor or not callable(preprocessor['extract_features']):
                            preprocessor['extract_features'] = extract_features
                        if 'clean_text' not in preprocessor or not callable(preprocessor['clean_text']):
                            preprocessor['clean_text'] = clean_text
                    else:
                        preprocessor = {
                            'extract_features': extract_features,
                            'clean_text': clean_text
                        }
                
                elif any(seq_type in model_name for seq_type in ["lstm", "gru", "transformer"]):
                    if isinstance(preprocessor, dict):
                        if 'clean_text' not in preprocessor or not callable(preprocessor['clean_text']):
                            preprocessor['clean_text'] = clean_text
                        
                        if 'tokenizer' not in preprocessor and 'max_seq_length' not in preprocessor:
                            if hasattr(preprocessor, 'word_index'):
                                tokenizer = preprocessor
                                preprocessor = {
                                    'tokenizer': tokenizer,
                                    'max_seq_length': 128,
                                    'clean_text': clean_text
                                }
                    else:
                        if hasattr(preprocessor, 'word_index'):
                            tokenizer = preprocessor
                            preprocessor = {
                                'tokenizer': tokenizer,
                                'max_seq_length': 128,
                                'clean_text': clean_text
                            }
                        else:
                            print(f"Creating new tokenizer for {model_name}")
                            tokenizer = Tokenizer(num_words=2500, oov_token="<OOV>")
                            preprocessor = {
                                'tokenizer': tokenizer,
                                'max_seq_length': 128,
                                'clean_text': clean_text
                            }
                
                elif "ensemble" in model_name:
                    if isinstance(preprocessor, dict):
                        if 'clean_text' not in preprocessor or not callable(preprocessor['clean_text']):
                            preprocessor['clean_text'] = clean_text
                        if 'extract_features' not in preprocessor or not callable(preprocessor['extract_features']):
                            preprocessor['extract_features'] = extract_features
                
                preprocessors[model_name] = preprocessor
                print(f"Successfully loaded and fixed preprocessor for {model_name}")
                
            except Exception as e:
                print(f"Error loading preprocessor for {model_name}: {str(e)}")
                if "dnn" in model_name:
                    preprocessors[model_name] = {
                        'extract_features': extract_features,
                        'clean_text': clean_text
                    }
                elif any(seq_type in model_name for seq_type in ["lstm", "gru", "transformer"]):
                    print(f"Creating new tokenizer for {model_name}")
                    tokenizer = Tokenizer(num_words=2500, oov_token="<OOV>")
                    preprocessors[model_name] = {
                        'tokenizer': tokenizer,
                        'max_seq_length': 128,
                        'clean_text': clean_text
                    }
        else:
            print(f"No preprocessor found for {model_name}, creating default preprocessor")
            if "dnn" in model_name:
                preprocessors[model_name] = {
                    'extract_features': extract_features,
                    'clean_text': clean_text
                }
            elif any(seq_type in model_name for seq_type in ["lstm", "gru", "transformer"]):
                print(f"Creating new tokenizer for {model_name}")
                tokenizer = Tokenizer(num_words=2500, oov_token="<OOV>")
                preprocessors[model_name] = {
                    'tokenizer': tokenizer,
                    'max_seq_length': 128,
                    'clean_text': clean_text
                }
    except Exception as e:
        print(f"Error loading {model_name}: {str(e)}")

print(f"\nLoaded {len(models)} models: {list(models.keys())}")

for model_name, preprocessor in preprocessors.items():
    if any(seq_type in model_name for seq_type in ["lstm", "gru", "transformer"]):
        if 'tokenizer' in preprocessor and not hasattr(preprocessor['tokenizer'], 'word_index'):
            print(f"Fitting tokenizer for {model_name} on teacher data")
            preprocessor['tokenizer'].fit_on_texts(teacher_data['Text'].values)


## Preprocess Data

In [ ]:
def preprocess_for_model(texts, model_name, preprocessor):
    
    if "dnn" in model_name:
        if 'extract_features' in preprocessor and 'clean_text' in preprocessor:
            cleaned_texts = [preprocessor['clean_text'](text) for text in texts]
            features = preprocessor['extract_features'](cleaned_texts)
            return features
        else:
            print(f"Missing required preprocessing functions for {model_name}")
            return None
    
    elif any(seq_type in model_name for seq_type in ["lstm", "gru", "transformer", "cnn"]):
        if 'tokenizer' in preprocessor and 'max_seq_length' in preprocessor and 'clean_text' in preprocessor:
            cleaned_texts = [preprocessor['clean_text'](text) for text in texts]
            sequences = preprocessor['tokenizer'].texts_to_sequences(cleaned_texts)
            padded = pad_sequences(sequences, maxlen=preprocessor['max_seq_length'], padding='post', truncating='post')
            return padded
        else:
            print(f"Missing required preprocessing components for {model_name}")
            return None
    
    elif "ensemble" in model_name:
        if isinstance(preprocessor, dict) and 'model_names' in preprocessor:
            inputs = []
            for i, input_type in enumerate(preprocessor['input_types']):
                if input_type == 'features':
                    cleaned_texts = [preprocessor['clean_text'](text) for text in texts]
                    features = preprocessor['extract_features'](cleaned_texts)
                    inputs.append(features)
                else:
                    cleaned_texts = [preprocessor['clean_text'](text) for text in texts]
                    sequences = preprocessor['tokenizer'].texts_to_sequences(cleaned_texts)
                    padded = pad_sequences(sequences, maxlen=preprocessor['max_seq_length'], padding='post', truncating='post')
                    inputs.append(padded)
            return inputs
        else:
            print(f"Invalid preprocessor format for {model_name}")
            return None
    
    else:
        print(f"Unknown model type: {model_name}")
        return None

processed_data = {}
for model_name, model in models.items():
    if model_name in preprocessors:
        print(f"Preprocessing data for {model_name}...")
        processed_data[model_name] = preprocess_for_model(teacher_data['Text'].values, model_name, preprocessors[model_name])
        
        if processed_data[model_name] is not None:
            if isinstance(processed_data[model_name], list):
                print(f"  Processed data shapes: {[x.shape for x in processed_data[model_name]]}")
            else:
                print(f"  Processed data shape: {processed_data[model_name].shape}")
    else:
        print(f"No preprocessor available for {model_name}, skipping...")


## Make Predictions

In [ ]:
predictions = {}
probabilities = {}

for model_name, model in models.items():
    if model_name in processed_data and processed_data[model_name] is not None:
        print(f"Making predictions with {model_name}...")
        
        try:
            if isinstance(processed_data[model_name], list):
                probs = model.predict(processed_data[model_name])
            else:
                probs = model.predict(processed_data[model_name])
            
            preds = (probs > 0.5).astype(int)
            
            predictions[model_name] = preds.flatten()
            probabilities[model_name] = probs.flatten()
            
            print(f"  Made {len(predictions[model_name])} predictions")
        except Exception as e:
            print(f"  Error making predictions with {model_name}: {str(e)}")

results_df = teacher_data[['ID', 'Text', 'Label', 'numeric_label']].copy()

for model_name in predictions.keys():
    results_df[f'{model_name}_prob'] = probabilities[model_name]
    results_df[f'{model_name}_pred'] = predictions[model_name]
    results_df[f'{model_name}_correct'] = results_df['numeric_label'] == results_df[f'{model_name}_pred']

print("\nSample of prediction results:")
display_cols = ['ID', 'Label'] + [col for col in results_df.columns if '_prob' in col or '_pred' in col or '_correct' in col]
print(results_df[display_cols].head())


## Evaluate Models

In [ ]:
accuracies = {}
for model_name in predictions.keys():
    accuracies[model_name] = accuracy_score(results_df['numeric_label'], results_df[f'{model_name}_pred'])
    print(f"{model_name} accuracy: {accuracies[model_name]:.4f}")

print("\nDetailed Classification Reports:")
for model_name in predictions.keys():
    print(f"\n{model_name.upper()} Classification Report:")
    print(classification_report(
        results_df['numeric_label'], 
        results_df[f'{model_name}_pred'],
        target_names=['Human', 'AI']
    ))
    
    cm = confusion_matrix(results_df['numeric_label'], results_df[f'{model_name}_pred'])
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=['Human', 'AI'],
        yticklabels=['Human', 'AI']
    )
    plt.title(f'{model_name.upper()} - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()


## Visualize Results

In [ ]:
plt.figure(figsize=(10, 6))
model_names = list(accuracies.keys())
acc_values = list(accuracies.values())

sorted_indices = np.argsort(acc_values)[::-1]
sorted_names = [model_names[i] for i in sorted_indices]
sorted_accs = [acc_values[i] for i in sorted_indices]

bars = plt.bar(sorted_names, sorted_accs, color='skyblue')

for bar, acc in zip(bars, sorted_accs):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.01,
        f'{acc:.4f}',
        ha='center',
        fontweight='bold'
    )

plt.title('Model Accuracy Comparison on Teacher Dataset')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.ylim(0, 1.1)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Conclusion

In [ ]:
print("\nModel Performance Summary:")
for model_name, accuracy in sorted(accuracies.items(), key=lambda x: x[1], reverse=True):
    print(f"{model_name}: {accuracy:.4f}")

if accuracies:
    best_model = max(accuracies.items(), key=lambda x: x[1])[0]
    print(f"\nBest performing model: {best_model} with accuracy {accuracies[best_model]:.4f}")
    
    if 'ensemble' in best_model:
        print("The ensemble model outperforms individual models, suggesting that combining models is effective.")
    else:
        print(f"The {best_model} model performs best on this dataset.")
else:
    print("No models were evaluated.")

